In [1]:
%load_ext autoreload
%autoreload 2

#### Importing

In [2]:
import json
from functools import partial
import numpy as np
import pandas as pd
import torch
from fancy_einsum import einsum

from transformers import AutoModelForCausalLM
from datasets import load_from_disk


import transformer_lens as tl
from circuitsvis.attention import attention_heads
import transformer_lens.utils as utils
from transformer_lens import ActivationCache, HookedTransformer, HookedTransformerConfig

from geomechinterp.causal.mygpt import SymbolTokenizer, DataCollator
from geomechinterp.tflens.utils import load_gpt2_to_hooked_transformer
from geomechinterp.causal.base_functions import ALL_BINARY_GENERATOR_FUNC_NAMES
from geomechinterp.causal.utils import DisplayChain

In [3]:
from IPython.display import HTML, IFrame
from tqdm import tqdm

import plotly.express as px
import plotly.graph_objects as go
from matplotlib import pyplot as plt
import plotly.io as pio

#### Loading Model

In [4]:
model_hf = AutoModelForCausalLM.from_pretrained("../gpt_rope_custom/checkpoint-407000/")


hook_config = HookedTransformerConfig(d_vocab=29, n_ctx=128, d_model=128,
                                      d_head=32, n_layers=6, n_heads=4,
                                      rotary_dim=64, act_fn='gelu_new', 
                                      original_architecture='GPT2LMHeadModel')

model = load_gpt2_to_hooked_transformer(model_hf, hook_config)

# model.tokenizer = SymbolTokenizer()

model_hf.to("mps").eval()
model.to("mps").eval()
print('')

Moving model to device:  mps



In [5]:
input_ids = torch.tensor([1, 2, 3, 3, 1, 2, 5, 6, 2, 10, 2, 3, 15, 18, 20, 7]).to("mps")

out1 = model_hf(input_ids)
out2, hooked_act = model.run_with_cache(input_ids)

k = 1
print('WARNING: SMALL DIFFERENCES IN ACTIVATIONS ARE ACCUMULATING, PROBABLY DUE TO DISREPANCIES IN THE LAYER NORM!')
print('---'*10)
for k in range(out2.shape[1]):
    # use cosine similarity after softmax
    print(torch.cosine_similarity(torch.nn.functional.softmax(out1['logits'][k,:], dim=-1), torch.nn.functional.softmax(out2[:,k,:], dim=-1)).item())    


------------------------------
0.9993540048599243
1.0
1.0
0.999977707862854
1.0
1.0
0.9522892236709595
1.0
0.9994153380393982
0.934735894203186
1.0
1.0
0.9226440191268921
0.9750189781188965
0.9235070943832397
0.9390167593955994


#### Loading Dataset

In [6]:
tokenizer = SymbolTokenizer()
dataset = load_from_disk("../tokenized_train")
test_dataset = load_from_disk("../tokenized_test")

train_collator = DataCollator(dataset, device="mps")  
test_collator = DataCollator(test_dataset, device="mps")  

In [7]:
# select subset of deterministic patterns
deterministic_train_dataset = []
for case in dataset:
    if case['stochastic']:
        continue
    else:
        deterministic_train_dataset.append(case)

deterministic_test_dataset = []
for case in test_dataset:
    if case['stochastic']:
        continue
    else:
        deterministic_test_dataset.append(case)

print(len(deterministic_train_dataset))
print(len(deterministic_test_dataset))

1011
46


In [8]:
from datasets import Dataset

deterministic_test_dataset = Dataset.from_list(deterministic_test_dataset)
deterministic_train_dataset = Dataset.from_list(deterministic_train_dataset)

# drop all rows with text length < 5
deterministic_train_dataset = deterministic_train_dataset.filter(lambda x: len(x['text']) > 5)
deterministic_test_dataset = deterministic_test_dataset.filter(lambda x: len(x['text']) > 5)

Filter:   0%|          | 0/1011 [00:00<?, ? examples/s]

Filter:   0%|          | 0/46 [00:00<?, ? examples/s]

In [9]:
idx = 2
test_case = deterministic_test_dataset[idx]
out = model(torch.tensor(test_case['input_ids']).to("mps"))
decoded_str = tokenizer.decode(out.argmax(dim=-1).tolist())[0]

print(DisplayChain.from_json(test_case['generator'], base_functions=ALL_BINARY_GENERATOR_FUNC_NAMES))
print(test_case['text'][6:])
print(decoded_str[5:])

Chain of functions:
DependentFeature(function=plus_minus_f, controls=[ab_prev, position_parity], truth_table=[1, 0, 1, 1])
DependentFeature(function=ab_f, controls=[+-, ab_prev, position_parity], truth_table=[1, 0, 1, 1, 1, 1, 1, 0])
DependentFeature(function=case_f, controls=[+-, ab, ab_prev], truth_table=[1, 0, 0, 0, 0, 0, 0, 1])
+b +b +a -a +b +b +a -a +b +b +a -a +b +b +a -a +b +b +a -a +b +b +a -a +b +b +a -a
+a +a -a +a +b -b -b +b +b -b -a +b +a +b +a +b +a +b +a +a +b +b -


#### Visualize MyGPT Activations in Reduced Dimensions

##### Deterministic Patterns Only (0.1% of the dataset)

In [11]:
# blocks.1.attn.hook_z
# blocks.2.hook_mlp_out
# blocks.2.hook_resid_post
# list(hooked_act.keys())

In [39]:
from geomechinterp.ds.emb_pipe import ActivationStatsPipeline   

selected_hooks = [f"blocks.{i}.hook_resid_post" for i in range(1, 6)]

emb_pipe = ActivationStatsPipeline(
    dataset=deterministic_train_dataset,
    activations_dir="tensors",
    activations_file_substring="train_deterministic",
    selected_hooks=selected_hooks,
)

emb_pipe.load_activations()

INFO:root:Using Lazy Loading of Activations from tensors
INFO:root:Built feature dataframe with 251 features
100%|██████████| 2/2 [00:00<00:00, 15.99it/s]
INFO:root:Loaded 5 hooked activations.
100%|██████████| 2/2 [00:00<00:00, 46.98it/s]
INFO:root:Loaded 5 hooked activations.


In [13]:
# target_layer = 'blocks.5.hook_resid_post'
# emb_pipe.cluster(activations_str=target_layer, agg_func=0, max_clusters=3, use_isomap=True)
# emb_pipe.plot_pca()

In [ ]:
emb_pipe.streamlit_viz()

In [37]:
results = emb_pipe.run_all()

In [52]:
emb_pipe.calc_activations_svd(cursor='blocks.2.hook_resid_post_4')

{'matrix_rank': 128,
 'spectral_ratio': 0.1749749,
 'frobenius_ratio': 0.6604135,
 'gini': 0.8232334033797917,
 'explained_variance_0.95': 10}

In [38]:
results

{'blocks.1.hook_resid_post_mean': {'clusters': array([0, 0, 1, ..., 0, 1, 1], dtype=int32),
  'matrix_rank': 128,
  'anova':                           Feature   F-statistic        p-value  \
  0                   num_functions  61805.554154   0.000000e+00   
  3                     presence_+-   1194.271040  1.631541e-266   
  4                     presence_><   1194.271040  1.631541e-266   
  5                     presence_][   1194.271040  1.631541e-266   
  6                     max_control    955.869104  1.993351e-233   
  16             tt_size_8_tt_sym_0    681.170826  7.948713e-188   
  18             tt_size_8_tt_sym_2    681.170826  7.948713e-188   
  17             tt_size_8_tt_sym_1    681.170826  7.948713e-188   
  15                tt_size_8_count    681.170826  7.948713e-188   
  1                       num_edges    645.133509  4.522302e-181   
  7    total_tt_symmetry_complexity    311.915746  3.778871e-106   
  9            pattern_entropy_char    241.234788   2.534599e

#### Try predicting Analytical Features from Activations

In [232]:
from geomechinterp.ds.cluster import calculate_feature_importance, single_position, mean_across_positions, agg_across_positions

# Aggregate activations and calculate feature importance for clustering
positions = [0, 1, 2, 3, 4]  # Available positions
activations_np = accumulated_activations['blocks.5.hook_resid_post'].detach().cpu().numpy()
binarized_features = cat_data.to_numpy()
feature_names = cat_data.columns

# IMPORTANT - FEATURES HERE ARE CHANNELS IN THE LLM ACTIVATIONS, NOT THE "ANALYTICAL" FEATURES OF THE PATTERNS
# SO IT IS ESSENTIALLY A VARIATION ON THE LINEAR PROBE TEST

position_importances = {}
for pos in positions:
    X = single_position(activations_np, pos)
    y = binarized_features[:, 0]  # Example target: first binary feature
    importance_df, _ = calculate_feature_importance(X, y)
    # rename features from idx to column names
    importance_df.rename(columns={0: 'Feature'}, inplace=True)
    position_importances[pos] = importance_df

# b) Mean across positions
X_mean = mean_across_positions(activations_np)
y_mean = binarized_features[:, 0]  # Example target
mean_importance_df, _ = calculate_feature_importance(X_mean, y_mean)


# c) Aggregate across positions
X_agg = agg_across_positions(activations_np)
y_agg = binarized_features[:, 0]  # Example target
agg_importance_df, _ = calculate_feature_importance(X_agg, y_agg)

In [233]:
print("Feature Importance for Single Positions:")
for pos, importance in position_importances.items():
    print(f"Position {pos}:")
    print(importance.sort_values(by='RandomForestImportance', ascending=False).head(3))

print("\nFeature Importance for Mean Across Positions:")
print(mean_importance_df.sort_values(by='RandomForestImportance', ascending=False).head(5))

print("\nFeature Importance for Aggregate Across Positions:")
print(agg_importance_df.sort_values(by='RandomForestImportance', ascending=False).head(10))

Feature Importance for Single Positions:
Position 0:
    Feature  RandomForestImportance  PermutationImportance
58       58                0.089877                    0.0
83       83                0.089353                    0.0
11       11                0.089277                    0.0
Position 1:
    Feature  RandomForestImportance  PermutationImportance
85       85                0.050026               0.000792
7         7                0.049765               0.000000
90       90                0.049748               0.000000
Position 2:
    Feature  RandomForestImportance  PermutationImportance
11       11                0.069365                    0.0
83       83                0.069357                    0.0
35       35                0.069061                    0.0
Position 3:
    Feature  RandomForestImportance  PermutationImportance
39       39                0.130574                    0.0
10       10                0.092375                    0.0
73       73               

### Example running on pattern

In [153]:
from geomechinterp.utils import run_model_on_pattern, run_model_on_pattern_and_plot

In [ ]:
pattern = 'a a a a a a a a a a a a a a a a a a a a a a a a a'
loss = run_model_on_pattern(model, pattern)
print(loss)

In [ ]:
pattern = 'b b b b b b b b b b b b b b b b b b b b b b b b b'
_ = run_model_on_pattern_and_plot(model, pattern)

In [ ]:
pattern = 'a b a b a b a b a b a b a b a b a b a b a b a b a b'
_ = run_model_on_pattern_and_plot(model, pattern, annotate_tokens=True)

In [ ]:
ALL_DETERMINISTIC_PATTERNS_LIST = list(ALL_DETERMINISTIC_PATTERNS_AND_GENERATORS2.keys())

losses = []
for pattern in ALL_DETERMINISTIC_PATTERNS_LIST:
    loss = run_model_on_pattern(model, pattern[2:], exclude_first_k=5)
    losses.append(loss)

df = pd.DataFrame({'pattern': [p[:29] for p in ALL_DETERMINISTIC_PATTERNS_LIST], 'loss': losses})